# Exposure-compensated 3DGS on a Colab GPU

This notebook trains plain 3DGS and the exposure-compensation code (`code-v3`) on a free
Colab GPU and produces the numbers and images for the paper.

**Before you start**

1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. On Google Drive, the folder `3dgs` must contain two files:
   * `code-v3.zip` (the newest one, it is next to this notebook on your Mac)
   * the dataset, `room.zip`
3. Menu **Runtime → Run all**. Nothing needs to be pasted or edited.

**If the runtime disconnects, or Colab says the GPU limit is reached,** little is lost:

* every finished run is copied to your Drive the moment it ends, and
* a run that was cut off left a checkpoint on Drive and continues from there.

Open the notebook again (when a GPU is available) and choose **Run all**.

Keep this browser tab open and the computer awake while it trains. When section 9 has
printed its table, choose **Runtime → Disconnect and delete runtime** so that no GPU
allowance is used up while nothing is running.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: use Runtime > Change runtime type > T4 GPU"

## 2. Connect Google Drive
A window will ask for permission to read your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Settings
Change the names here if yours are different.

In [ ]:
DRIVE_DIR   = "/content/drive/MyDrive/3dgs"   # the folder you made on Drive
CODE_ZIP    = "code-v3.zip"
DATA_ZIP    = "room.zip"                       # your dataset zip

ITERATIONS  = 30000   # full-length training, as in the paper
RESOLUTION  = 2       # half size. Every run that is compared must use the same value
DATA_DEVICE = "cuda"  # "cpu" keeps the photos out of GPU memory (slower)
OUT_DIR     = "/content/output"

# The full-length runs of section 8, in this order: (name, method, extra options).
#   "trainv3"  = code-v3 (plain 3DGS with --no_exposure, otherwise the exposure method)
#   "3dgs2024" = the official 2024 version of 3DGS
#   "v1"       = the original trainv1.py
# the exposure-compensation options from the README of the 2024 code
EXP2024 = "--train_test_exp --exposure_lr_init 0.001 --exposure_lr_final 0.0001 --exposure_lr_delay_steps 5000 --exposure_lr_delay_mult 0.001"
RUNS = [
    ("plain",          "trainv3",  "--no_exposure"),              # ordinary 3DGS, the baseline
    ("exposure_const", "trainv3",  "--exposure_init constant"),   # exposure method, every exposure starts at 0.95
    ("exposure",       "trainv3",  ""),                           # exposure method, exposures start from the grey levels
    ("3dgs2024",       "3dgs2024", ""),                           # 3DGS 2024 with its default options
    ("v1",             "v1",       ""),                           # the first version of the exposure code
    ("3dgs2024_exp",   "3dgs2024", EXP2024),                      # 3DGS 2024 with its exposure compensation switched on
]

def on_drive(name):
    return os.path.isfile(f"{DRIVE_DIR}/results_{name}.zip")

def pending(method):
    """True if a run of this method still has to be trained."""
    return any(m == method and not on_drive(n) for n, m, _ in RUNS)

import os
for f in (CODE_ZIP, DATA_ZIP):
    assert os.path.isfile(f"{DRIVE_DIR}/{f}"), f"Not found on Drive: {DRIVE_DIR}/{f}"
print("Both files found.")
for name, method, _ in RUNS:
    print(f"  run '{name}': {'already on Drive, will be skipped' if on_drive(name) else 'to do'}")

## 4. Install gaussian-splatting (the 2023 version)
The exposure code is written for the original 2023 release, so this checks out the last
commit of 2023 instead of the newest version. Takes a few minutes.

In [ ]:
%cd /content
!rm -rf gaussian-splatting
!git clone -q https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /content/gaussian-splatting
!git checkout -q 2eee0e26d2d5fd00ec462df47752223952f6bf4e
!git submodule update -q --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn

# Newer compilers need these two headers to be named explicitly.
!grep -q "<cstdint>" submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h || sed -i '1i #include <cstdint>' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
!grep -q "<cfloat>" submodules/simple-knn/simple_knn.cu || sed -i '1i #include <cfloat>' submodules/simple-knn/simple_knn.cu

!pip install -q plyfile
# --no-build-isolation: these two packages need the PyTorch that is already installed in order to build.
!pip install --no-build-isolation ./submodules/diff-gaussian-rasterization ./submodules/simple-knn 2>&1 | tail -n 25
import importlib
importlib.invalidate_caches()
for m in ("diff_gaussian_rasterization", "simple_knn._C"):
    importlib.import_module(m)
print("Rasteriser and simple-knn installed.")

## 5. Add the exposure code and run its self-test
All 8 tests should say PASS.

In [ ]:
import glob, shutil, zipfile
shutil.rmtree("/content/code_v3_tmp", ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_DIR}/{CODE_ZIP}") as z:
    z.extractall("/content/code_v3_tmp")
found = glob.glob("/content/code_v3_tmp/**/trainv3.py", recursive=True)
assert found, "trainv3.py is not inside the code zip"
src = os.path.dirname(found[0])
assert os.path.isfile(f"{src}/baselines/v1_original/render_v1.py"), \
    "The code-v3.zip on Drive is an old one. Replace it with the code-v3.zip next to this notebook on your Mac, then run this cell again."
repo = "/content/gaussian-splatting"
for name in ("trainv3.py", "render_exposure.py"):
    shutil.copy(f"{src}/{name}", repo)
for name in ("exposure", "tests"):
    shutil.rmtree(f"{repo}/{name}", ignore_errors=True)
    shutil.copytree(f"{src}/{name}", f"{repo}/{name}")
CODE_SRC = src
%cd /content/gaussian-splatting
!python tests/test_exposure_logic.py

### 5b. The two comparison methods
**v1** (the first version of the exposure code) runs in its own copy of the 2023 repository,
because its `exposure` folder has the same name as the new one. `trainv1.py` is used as it
is, with one added line that also saves its exposure values, which the evaluation needs.

**3DGS 2024** is downloaded from the official repository (commit 54c035f, 30 October 2024).
It needs its own rasteriser, which is installed in a separate folder so that it does not
replace the 2023 one. Both steps are skipped when those runs are already on Drive.

In [ ]:
V1_REPO, REPO2024, RAST2024 = "/content/gs_v1", "/content/gs2024", "/content/rast2024"
COMMIT2024 = "54c035f7834b564019656c3e3fcc3646292f727d"
# The 2024 code is run with this search path, so that it finds its own rasteriser first.
PYPATH2024 = RAST2024 + ":" + os.environ.get("PYTHONPATH", "")

if pending("v1"):
    shutil.rmtree(V1_REPO, ignore_errors=True)
    shutil.copytree("/content/gaussian-splatting", V1_REPO,
                    ignore=shutil.ignore_patterns("exposure", "tests", "trainv3.py", "render_exposure.py",
                                                  "output", ".git", "build", "*.egg-info"))
    v1_src = f"{CODE_SRC}/baselines/v1_original"
    shutil.copytree(f"{v1_src}/exposure", f"{V1_REPO}/exposure")
    shutil.copy(f"{v1_src}/render_v1.py", V1_REPO)
    text = open(f"{v1_src}/trainv1.py").read()
    anchor = "                gaussians.save_ply(ply_file_path)\n"
    assert text.count(anchor) == 1, "trainv1.py is not the file this notebook expects"
    text = text.replace(anchor, anchor +
        '                torch.save(exposure_model.state_dict(), os.path.join(point_cloud_path, "exposure_model.pt"))  # added: needed to evaluate\n')
    open(f"{V1_REPO}/trainv1.py", "w").write(text)
    print("v1 is ready in", V1_REPO)

if pending("3dgs2024"):
    !rm -rf {REPO2024} {RAST2024}
    !git clone -q https://github.com/graphdeco-inria/gaussian-splatting.git {REPO2024}
    !cd {REPO2024} && git checkout -q {COMMIT2024} && git submodule update -q --init --recursive submodules/diff-gaussian-rasterization
    !cd {REPO2024} && (grep -q "<cstdint>" submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h || sed -i '1i #include <cstdint>' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h)
    !cd {REPO2024} && pip install --no-build-isolation --target {RAST2024} ./submodules/diff-gaussian-rasterization 2>&1 | tail -n 15
    # the 2024 rasteriser has an anti-aliasing setting that the 2023 one does not have
    import subprocess, sys
    check = "from diff_gaussian_rasterization import GaussianRasterizationSettings as S; print('antialiasing' in S._fields)"
    new = subprocess.run([sys.executable, "-c", check], env=dict(os.environ, PYTHONPATH=PYPATH2024), capture_output=True, text=True)
    old = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
    assert new.stdout.strip() == "True", "The 2024 rasteriser did not install:\n" + new.stdout + new.stderr
    assert old.stdout.strip() == "False", "The 2023 rasteriser was replaced:\n" + old.stdout + old.stderr
    print("2024 rasteriser installed next to the 2023 one.")

## 6. Unpack the dataset
This also estimates how much memory the photos need.

In [ ]:
from PIL import Image
shutil.rmtree("/content/data", ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_DIR}/{DATA_ZIP}") as z:
    z.extractall("/content/data")
cands = sorted(os.path.dirname(p) for p in glob.glob("/content/data/**/sparse", recursive=True))
assert cands, "No 'sparse' folder in the zip. The zip must contain images/ and sparse/0/ (see section 6b)."
DATA = cands[0]
imgs = sorted(glob.glob(f"{DATA}/images/*"))
assert imgs, f"No photos in {DATA}/images"
w, h = Image.open(imgs[0]).size
if RESOLUTION in (2, 4, 8):
    w, h = w // RESOLUTION, h // RESOLUTION
elif RESOLUTION == -1 and w > 1600:
    w, h = 1600, int(h * 1600 / w)
gb = len(imgs) * w * h * 3 * 4 / 1e9
print(f"Dataset: {DATA}")
print(f"{len(imgs)} photos, trained at {w}x{h}; about {gb:.1f} GB to hold them in memory")
if gb > 6:
    print(">> That is a lot for a free Colab GPU. In section 3 either set RESOLUTION = 2 (half size, faster),")
    print(">> or keep full size with DATA_DEVICE = \"cpu\" (slower). Then run from section 3 again.")

### 6b. Only if you do not have `sparse/0` (optional)
If all you have is the photos, put them in a zip, and set `RUN_COLMAP = True` below. This
runs COLMAP on Colab's CPU, which is slow (possibly more than an hour for a few hundred
photos) and gives slightly different camera poses from your earlier dataset, so the old
baseline numbers would no longer be directly comparable.

In [ ]:
RUN_COLMAP = False
if RUN_COLMAP:
    !apt-get -qq install -y colmap > /dev/null
    photos = [p for p in glob.glob("/content/data/**/*", recursive=True)
              if p.lower().endswith((".jpg", ".jpeg", ".png"))]
    assert photos, "No photos found in the zip"
    os.makedirs("/content/data_colmap/input", exist_ok=True)
    for p in photos:
        shutil.copy(p, "/content/data_colmap/input")
    !python convert.py -s /content/data_colmap --no_gpu
    DATA = "/content/data_colmap"
    print("Dataset:", DATA)

## 7. Short tests (a few minutes)
Each method that still has a run to do is tried for 700 iterations first, so that a problem
shows up now and not after hours of training:

* the exposure code is trained twice, once straight through and once continued from a
  checkpoint written at iteration 400 (the two PSNR values should be close to each other);
* 3DGS 2024 and v1 are trained and their test views rendered.

The PSNR values printed here are low; 700 iterations is only the very beginning of training.
**If this cell ends with an error, stop here and send me the text it printed.**

In [ ]:
import json, re, time

REPO = "/content/gaussian-splatting"
CKPT_ITERS = "15000 23000"      # where a full-length run leaves a checkpoint on Drive
os.makedirs(OUT_DIR, exist_ok=True)

def sh(cmd):
    get_ipython().system(cmd)

def save_to_drive(name, save_models=False):
    """Zip the test renders, exposure maps, metrics, settings and log of one run onto Drive."""
    root = f"{OUT_DIR}/{name}"
    tmp = f"{OUT_DIR}/results_{name}.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for folder, _, files in os.walk(root):
            rel = os.path.relpath(folder, root)
            if rel.startswith("train"):
                continue
            if rel.startswith("point_cloud") and not save_models:
                files = [f for f in files if f.endswith(".pt")]
            for f in files:
                if f.startswith("events.out") or f.endswith(".pth") or f == "train_log_raw.txt":
                    continue
                z.write(os.path.join(folder, f), os.path.normpath(os.path.join(name, rel, f)))
    out = f"{DRIVE_DIR}/results_{name}.zip"
    shutil.copy(tmp, out + ".part")
    os.replace(out + ".part", out)
    print(f"[{name}] saved to Drive: {out} ({os.path.getsize(tmp) / 1e6:.0f} MB)")

def usable_checkpoint(name):
    """A local copy of the checkpoint that a cut-off run left on Drive, or None."""
    src = f"{DRIVE_DIR}/checkpoint_{name}/checkpoint.pth"
    if not os.path.isfile(src):
        return None
    dst = f"{OUT_DIR}/resume_{name}.pth"
    try:
        shutil.copyfile(src, dst)
        with zipfile.ZipFile(dst) as z:
            assert z.testzip() is None, "damaged file"
        return dst
    except Exception as exc:
        print(f"[{name}] the checkpoint on Drive cannot be used ({exc}); starting from the beginning.")
        return None

def train_commands(method, root, iterations, options, checkpoint=""):
    """The three shell commands (train, render the test views, metrics) of one method."""
    common = (f'-s "{DATA}" -m {root} --eval -r {RESOLUTION} --data_device {DATA_DEVICE} '
              f'--iterations {iterations} --test_iterations 7000 {iterations} --save_iterations {iterations}')
    if method == "trainv3":
        train = f'cd {REPO} && python -u trainv3.py {common} {checkpoint or "--checkpoint_iterations 0"} {options}'
        render = f'cd {REPO} && python render_exposure.py -m {root} --skip_train'
    elif method == "v1":
        train = f'cd {V1_REPO} && python -u trainv1.py {common} --checkpoint_iterations 0 {options}'
        render = f'cd {V1_REPO} && python render_v1.py -m {root} --skip_train'
    elif method == "3dgs2024":
        env = f'PYTHONPATH={PYPATH2024}'
        train = f'cd {REPO2024} && {env} python -u train.py {common} --disable_viewer {options}'
        render = f'cd {REPO2024} && {env} python render.py -m {root} --skip_train'
    else:
        raise ValueError(f"unknown method '{method}'")
    # every method is scored by the same metrics script, the one of the 2023 repository
    return train, render, f'cd {REPO} && python metrics.py -m {root}'

def run_model(name, method, options):
    """Train (continuing from a checkpoint if there is one), render the test views,
    compute the metrics and save everything to Drive."""
    if on_drive(name):
        print(f"[{name}] already on Drive, skipped. Delete results_{name}.zip on Drive to train it again.")
        return
    root = f"{OUT_DIR}/{name}"
    ckpt_dir = f"{DRIVE_DIR}/checkpoint_{name}"
    shutil.rmtree(root, ignore_errors=True)
    os.makedirs(root)
    resume, checkpoint = None, ""
    if method == "trainv3":                # only the new code can continue from a checkpoint
        resume = usable_checkpoint(name)
        checkpoint = f"--checkpoint_iterations {CKPT_ITERS} --checkpoint_copy_dir {ckpt_dir}"
        if resume:
            checkpoint += f" --start_checkpoint {resume}"
    train, render, metrics = train_commands(method, root, ITERATIONS, options, checkpoint)
    print("=" * 30, name, f"({method})", options, "- continuing from a checkpoint" if resume else "", "=" * 30)
    t0 = time.time()
    sh(f'{train} 2>&1 | tee {root}/train_log_raw.txt')
    minutes = (time.time() - t0) / 60
    ply = f"{root}/point_cloud/iteration_{ITERATIONS}/point_cloud.ply"
    assert os.path.isfile(ply), f"[{name}] training did not finish; send me the output above."
    sh(render)
    sh(metrics)
    assert os.path.isfile(f"{root}/results.json"), f"[{name}] the metrics did not finish; send me the output above."

    # Keep the informative lines of the log (not the progress bars) and a few facts about the run.
    raw = open(f"{root}/train_log_raw.txt", errors="replace").read()
    lines = [l.strip() for l in re.split(r"[\r\n]+", raw)
             if l.strip() and "Training progress" not in l and "Reading camera" not in l
             and "it/s" not in l and "s/it" not in l]
    open(f"{root}/train_log.txt", "w").write("\n".join(lines) + "\n")
    with open(ply, "rb") as f:
        n_gauss = int(re.search(r"element vertex (\d+)", f.read(4000).decode("latin1")).group(1))
    resumed = re.search(r"Resumed from \S+ at iteration (\d+)", raw)
    info = {"name": name, "method": method, "options": options, "iterations": ITERATIONS,
            "resolution": RESOLUTION, "train_minutes": round(minutes, 1), "gaussians": n_gauss,
            "resumed_from_iteration": int(resumed.group(1)) if resumed else 0,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}
    json.dump(info, open(f"{root}/run_info.json", "w"), indent=1)
    print(f"[{name}] {n_gauss:,} Gaussians, this session trained for {minutes:.0f} min"
          + (f" (continued from iteration {info['resumed_from_iteration']})" if resumed else ""))
    print(open(f"{root}/results.json").read())
    save_to_drive(name)
    shutil.rmtree(ckpt_dir, ignore_errors=True)      # the checkpoint is no longer needed
    if resume:
        os.remove(resume)

def short_run(label, command, must_exist):
    """Run one command of the short tests; on failure show the end of its output and stop."""
    log = f"{OUT_DIR}/short_test/" + re.sub(r"\W+", "_", label) + ".log"
    print("-----", label)
    sh(f'{command} 2>&1 | tee {log} | grep -E "Evaluating test|Resumed|Checkpoint copied|Exposure values"')
    if not os.path.exists(must_exist):
        print("".join(open(log, errors="replace").read().replace("\r", "\n").splitlines(keepends=True)[-40:]))
        raise RuntimeError(f"The short test '{label}' did not finish; send me the text above.")

RUN_SHORT_TEST = True
if RUN_SHORT_TEST:
    t = f"{OUT_DIR}/short_test"
    shutil.rmtree(t, ignore_errors=True)
    os.makedirs(t)
    done700 = "point_cloud/iteration_700/point_cloud.ply"
    if pending("trainv3"):
        opts = "--exposure_init constant --exposure_warmup 100"
        train, _, _ = train_commands("trainv3", f"{t}/full", 700, opts,
                                     f"--checkpoint_iterations 400 --checkpoint_copy_dir {t}/copy")
        short_run("exposure code, straight through", train, f"{t}/full/{done700}")
        train, _, _ = train_commands("trainv3", f"{t}/resumed", 700, opts,
                                     f"--checkpoint_iterations 0 --start_checkpoint {t}/copy/checkpoint.pth")
        short_run("exposure code, continued from the checkpoint", train, f"{t}/resumed/{done700}")
    for name, method, options in RUNS:
        if method in ("v1", "3dgs2024") and not on_drive(name):
            train, render, _ = train_commands(method, f"{t}/{name}", 700, options)
            short_run(f"{name}, training", train, f"{t}/{name}/{done700}")
            short_run(f"{name}, rendering", render, f"{t}/{name}/test/ours_700/renders/00000.png")
    print("Short tests passed.")

## 8. The full-length runs
Trains each entry of `RUNS` (section 3), one after the other, on the same photos, the same
train/test split and the same image size, and scores all of them with the same metrics
script. Runs already on Drive are skipped, and each run is copied to Drive as soon as it
finishes.

The exposure runs leave a checkpoint of about 2 GB in `3dgs/checkpoint_<name>` on Drive at
iterations 15,000 and 23,000, so that a run that is cut off continues from there next time.
The checkpoint is deleted when the run finishes; if Drive has too little free space it is
simply not written. 3DGS 2024 and v1 cannot be continued and start again if they are cut off.
Expect about an hour for each of plain 3DGS, the two 3DGS 2024 runs and v1, and longer for
each exposure run.

**About 3DGS 2024.** It is trained twice. `3dgs2024` uses its default options. With those,
the exposure compensation of the 2024 code is not applied; the code only uses it with the
option `--train_test_exp`. `3dgs2024_exp` switches it on. That procedure trains on the left
half of every test photo and scores only the right half, so the numbers of `3dgs2024_exp`
are for right halves. The other methods are re-scored on the same halves afterwards, from
the saved test renders, before they are compared with it.

In [ ]:
for name, method, options in RUNS:
    run_model(name, method, options)

## 9. Results in the format of the paper
Mean ± standard deviation and range of PSNR, SSIM and LPIPS over the test images, for every
run that is on Drive. Also saved as `summary.txt` in the Drive folder.

In [ ]:
import statistics

def read_from_zip(z, filename):
    names = [n for n in z.namelist() if n.split("/")[-1] == filename]
    if not names:
        return None
    return json.loads(z.read(min(names, key=lambda n: n.count("/"))))

order = [n for n, *_ in RUNS]
zips = sorted(glob.glob(f"{DRIVE_DIR}/results_*.zip"),
              key=lambda p: (order.index(os.path.basename(p)[8:-4]) if os.path.basename(p)[8:-4] in order else 99, p))
report = []
for path in zips:
    name = os.path.basename(path)[8:-4]
    with zipfile.ZipFile(path) as z:
        per_view, info = read_from_zip(z, "per_view.json"), read_from_zip(z, "run_info.json")
    if not per_view:
        continue
    for method, metrics in per_view.items():
        report.append("=" * 60)
        report.append(f"Model: {name}   ({method}, {len(metrics['PSNR'])} test images)")
        if info:
            report.append(f"method: {info.get('method', 'trainv3')}   options: '{info['options']}'   resolution: 1/{info['resolution']}   "
                          f"Gaussians: {info['gaussians']:,}   training: {info['train_minutes']} min on {info['gpu']}")
        for key in ("PSNR", "SSIM", "LPIPS"):
            vals = metrics[key]
            v = list(vals.values())
            best = max(vals, key=vals.get) if key != "LPIPS" else min(vals, key=vals.get)
            worst = min(vals, key=vals.get) if key != "LPIPS" else max(vals, key=vals.get)
            report.append(f"{key}: {statistics.mean(v):.4f} +/- {statistics.pstdev(v):.4f}   "
                          f"range [{min(v):.4f}, {max(v):.4f}]   best {best}   worst {worst}")
report = "\n".join(report) if report else "No finished run on Drive yet."
print(report)
open(f"{DRIVE_DIR}/summary.txt", "w").write(report + "\n")
print("\nSaved", f"{DRIVE_DIR}/summary.txt")

## What to send back
* the text printed in section 9 (also saved as `summary.txt` on Drive)
* the `results_*.zip` files from your Drive folder (give Drive a minute to finish uploading
  before you close the tab)

Then choose **Runtime → Disconnect and delete runtime**. Deleted checkpoints stay in the Drive
bin for 30 days and count towards your storage; empty the bin if you need the space.

## Optional: one more run per component
Add a line to `RUNS` in section 3 and choose **Run all** again, for example
`("no_smooth", "trainv3", "--lambda_smooth 0")`,
`("no_consistency", "trainv3", "--lambda_consistency 0")`,
`("densify_x5", "trainv3", "--densify_scale 5")` or `("no_reset", "trainv3", "--no_opacity_reset")`.
To keep the trained point clouds as well, call `save_to_drive("exposure", save_models=True)`
in a new cell (a few hundred MB each).